In [ ]:

PROFILE   = "STANDARD"
TIME_BUDGET_MIN = 240
USE_DRIVE = True
RUN_SENS   = True
RUN_BLOCK  = True
RUN_AR     = True
RUN_BUDGET = True
RUN_SMOKE  = True
REUSE_V5   = True
DELETE_CHECKPOINTS_AFTER = False

import os, sys, time, zlib, gc, pickle, itertools, warnings, subprocess, shutil, io, traceback
from contextlib import contextmanager
warnings.filterwarnings("ignore"); os.environ["PYTHONWARNINGS"]="ignore"
for _m in ["shap","lime","catboost","xgboost","openpyxl","ucimlrepo","statsmodels","threadpoolctl"]:
    try: __import__(_m)
    except ImportError: subprocess.run([sys.executable,"-m","pip","-q","install",_m],check=False)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from scipy.signal import lfilter
from joblib import Parallel, delayed
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split, cross_val_predict
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import shap
from lime.lime_tabular import LimeTabularExplainer
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

CODE_VER="v6"; T_START=time.time()

OUT="/content/xai_ees"
if USE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
        if not os.path.exists("/content/drive/MyDrive"): raise RuntimeError("MyDrive bulunamadı")
        OUT="/content/drive/MyDrive/xai_ees"
    except Exception as e:
        raise RuntimeError("Drive bağlanamadı; önce AYRI bir hücrede şunu çalıştırıp izni onayla: "
                           "from google.colab import drive; drive.mount('/content/drive')  -> sonra bu hücreyi tekrar çalıştır. "
                           "(Geçici disk için USE_DRIVE=False yap; oturum kapanırsa her şey silinir.)") from e
CK_NEW=f"{OUT}/_checkpoints/v6_{PROFILE}"; CK_OLD=f"{OUT}/_checkpoints/v5_{PROFILE}"
RES=f"{OUT}/RESULTS_{PROFILE}"; FIG=f"{RES}/figures"; CSV=f"{RES}/csv"; DAT=f"{OUT}/data"
for d in (CK_NEW,FIG,CSV,DAT): os.makedirs(d,exist_ok=True)
CK_MAIN=CK_OLD if (REUSE_V5 and os.path.isdir(CK_OLD) and len(os.listdir(CK_OLD))>0) else CK_NEW
CKD={"sim":CK_MAIN,"real":CK_MAIN,"sens":CK_MAIN,"budget":CK_NEW,"block":CK_NEW,"ar":CK_NEW}


PROF={
 "TEST":    dict(sizes=[20,100,300],reps=2,n_test=300,n_expl=6,lime_n=150,kshap=200,bg=8,self_reps=1,sens_sizes=[100],sens_reps=2,real_reps=2,bj=[100,300],ff=[50,None],big=False,
                 b_sizes=[100],b_reps=2,b_lime=300,b_kshap=400,b_expl=8,b_bg=15,block_reps=2,ar_sizes=[100],ar_reps=2,ar_phis=[0.8]),
 "STANDARD":dict(sizes=[20,100,200,500,2000],reps=15,n_test=1000,n_expl=12,lime_n=400,kshap=600,bg=15,self_reps=3,sens_sizes=[100,500],sens_reps=8,real_reps=8,bj=[100,500,2000],ff=[50,100,None],big=False,
                 b_sizes=[100,500],b_reps=6,b_lime=1000,b_kshap=1500,b_expl=24,b_bg=30,block_reps=8,ar_sizes=[100,500,2000],ar_reps=8,ar_phis=[0.5,0.8]),
 "FINAL":   dict(sizes=[20,50,100,200,500,2000],reps=50,n_test=1000,n_expl=30,lime_n=1000,kshap=1000,bg=20,self_reps=5,sens_sizes=[100,500,2000],sens_reps=20,real_reps=20,bj=[50,100,500,2000],ff=[50,100,200,None],big=True,
                 b_sizes=[100,500,2000],b_reps=15,b_lime=1500,b_kshap=2000,b_expl=40,b_bg=40,block_reps=15,ar_sizes=[100,500,2000],ar_reps=15,ar_phis=[0.5,0.8]),
}[PROFILE]
CFG=dict(**PROF,ck=CK_MAIN)
CFG_B=dict(CFG); CFG_B.update(lime_n=PROF["b_lime"],kshap=PROF["b_kshap"],n_expl=PROF["b_expl"],bg=PROF["b_bg"],tree_int=True,ck=CK_NEW)
CFGK={"sim":CFG,"real":CFG,"sens":CFG,"budget":CFG_B,"block":CFG,"ar":CFG}
N_JOBS=max(1,os.cpu_count() or 1)
print(f"Kod {CODE_VER} | profil={PROFILE} | çekirdek={N_JOBS} | sonuç klasörü={RES}")
print(f"Ana kayıtlar (sim/real/sens): {CK_MAIN}  | yeni kayıtlar: {CK_NEW}")

sns.set_theme(style="whitegrid",context="paper",palette="colorblind",font_scale=1.15)
plt.rcParams.update({"figure.dpi":110,"axes.titleweight":"bold","axes.spines.top":False,"axes.spines.right":False})
MODEL_ORDER=["LogReg","XGBoost","CatBoost","RandomForest","ExtraTrees","SVM","ANN"]
PAL=dict(zip(MODEL_ORDER,sns.color_palette("colorblind",7)))
GUARD_ERR=[]

@contextmanager
def guard(label):
    try: yield
    except Exception as e:
        print(f"\n⚠ [{label}] ATLANDI -> {type(e).__name__}: {str(e)[:300]}"); GUARD_ERR.append((label,traceback.format_exc()[-900:])); plt.close("all")
def sv(fig,name):
    fig.savefig(f"{FIG}/{name}.png",dpi=300,bbox_inches="tight"); fig.savefig(f"{FIG}/{name}.pdf",bbox_inches="tight"); plt.show(); plt.close(fig)
def show(title,df,rows=40):
    print("\n"+"="*100+f"\n{title}\n"+"="*100)
    if df is not None and len(df): display(df.head(rows))
    else: print("(boş)")
def flat(df):
    df=df.copy()
    if isinstance(df.columns,pd.MultiIndex): df.columns=["_".join(map(str,c)) for c in df.columns]
    df.columns=[str(c) for c in df.columns]; return df
def save_tables(sheets,xlsx,prefix):
    with pd.ExcelWriter(xlsx,engine="openpyxl") as w:
        for k,v in sheets.items():
            if v is None or len(v)==0: continue
            v=flat(v)
            try: v.to_csv(f"{CSV}/{prefix}_{k}.csv",index=False)
            except Exception as e: print("CSV yazılamadı",k,e)
            try:
                if len(v)<1_000_000: v.to_excel(w,sheet_name=k[:31],index=False)
            except Exception as e: print("Excel sayfası yazılamadı",k,e)
    print("Excel kaydedildi:",xlsx)


POLL=["PM2.5","PM10","NO2","SO2","CO"]
OTHER=["WindSpeed","GreenSpace","Humidity","SolarRad","Precip","TrafficDensity","PopDensity","Temp","Pressure","IndustryIdx","DistHighway"]
NOISE=[f"Noise{i}" for i in range(1,11)]
ALL_FEATS=POLL+OTHER+NOISE
TRUE=["PM2.5","NO2","WindSpeed","TrafficDensity","GreenSpace"]; DECOY=["PM10","SO2","CO"]
NUIS=[f for f in OTHER if f not in TRUE]
COEF={"PM2.5":1.8,"NO2":1.5,"WindSpeed":-1.2,"TrafficDensity":1.3,"GreenSpace":-1.0}; BETA_SQ=-0.5
TRUE_W=np.array([abs(COEF.get(f,0.0)) for f in ALL_FEATS])
CAT_SIM={**{f:"true" for f in TRUE},**{f:"decoy" for f in DECOY},**{f:"nuisance" for f in NUIS},**{f:"noise" for f in NOISE}}
CATS_SIM=["true","decoy","nuisance","noise"]
BASE=dict(poll_shift=0.0,traffic_scale=1.0,green=(2,3),wind_scale=4.0,industry=0.0,b0=-0.3,rho=0.6,signal=1.0)
SCENARIOS={"Normal":BASE,
 "HighRisk":{**BASE,"poll_shift":0.25,"traffic_scale":1.4,"green":(1.5,4),"wind_scale":3.5,"industry":1.0},
 "LowRisk":{**BASE,"poll_shift":-0.25,"traffic_scale":0.7,"green":(4,2),"wind_scale":4.5,"industry":-1.0}}
SENS={"Rho0.2":{**BASE,"rho":0.2},"Rho0.9":{**BASE,"rho":0.9},"WeakSignal":{**BASE,"signal":0.4},"LowPrev":{**BASE,"b0":-3.0}}
SCENARIOS.update(SENS); MAIN_SC=["Normal","HighRisk","LowRisk"]
for _ph in (0.5,0.8): SCENARIOS[f"AR{_ph}"]={**BASE,"phi":_ph}
AR_SC=[f"AR{p}" for p in PROF["ar_phis"]]

def gen_raw(n,sc,rng):
    p=SCENARIOS[sc]
    if p.get("phi",0)>0: return gen_ar(n,sc,rng)
    k=5; s2=0.25
    cov=np.full((k,k),p["rho"]*s2); np.fill_diagonal(cov,s2)
    poll=np.exp(rng.multivariate_normal(np.array([3.0,3.5,3.0,2.0,0.5])+p["poll_shift"],cov,size=n))
    d={f:poll[:,j] for j,f in enumerate(POLL)}
    d["WindSpeed"]=p["wind_scale"]*rng.weibull(2.0,n); d["GreenSpace"]=rng.beta(*p["green"],n); d["Humidity"]=rng.beta(5,3,n)
    d["SolarRad"]=rng.gamma(4,50,n); d["Precip"]=rng.gamma(1.5,2,n); d["TrafficDensity"]=rng.gamma(3,100*p["traffic_scale"],n)
    d["PopDensity"]=rng.lognormal(7,1,n); d["Temp"]=rng.normal(15,8,n); d["Pressure"]=rng.normal(1013,8,n)
    d["IndustryIdx"]=rng.normal(50+10*p["industry"],10,n); d["DistHighway"]=rng.exponential(5,n)
    for i in range(1,11): d[f"Noise{i}"]=rng.normal(size=n)
    return pd.DataFrame(d)[ALL_FEATS]

def _ar_latent(n,k,phi,rng,corr=None,burn=60):
    """Durağan AR(1) Gauss süreci (birim varyans); değişkenler arası korelasyon corr (innovasyonlar üzerinden)."""
    L=np.linalg.cholesky(corr) if corr is not None else np.eye(k)
    e=rng.standard_normal((n+burn,k))@L.T
    z=lfilter([np.sqrt(1-phi**2)],[1.0,-phi],e,axis=0)
    return z[burn:]
def gen_ar(n,sc,rng):
    """Zamansal otokorelasyonlu DGP: aynı marjinaller (Gauss kopulası), AR(1) bağımlılık; gürültü değişkenleri de otokorelasyonlu."""
    p=SCENARIOS[sc]; phi=p["phi"]; k=5
    Sig=np.full((k,k),p["rho"]); np.fill_diagonal(Sig,1.0)
    Zp=_ar_latent(n,k,phi,rng,Sig); mu=np.array([3.0,3.5,3.0,2.0,0.5])+p["poll_shift"]
    poll=np.exp(mu+0.5*Zp)
    d={f:poll[:,j] for j,f in enumerate(POLL)}
    U=np.clip(stats.norm.cdf(_ar_latent(n,7,phi,rng)),1e-12,1-1e-12)
    d["WindSpeed"]=stats.weibull_min.ppf(U[:,0],2.0,scale=p["wind_scale"])
    d["GreenSpace"]=stats.beta.ppf(U[:,1],*p["green"]); d["Humidity"]=stats.beta.ppf(U[:,2],5,3)
    d["SolarRad"]=stats.gamma.ppf(U[:,3],4,scale=50); d["Precip"]=stats.gamma.ppf(U[:,4],1.5,scale=2)
    d["TrafficDensity"]=stats.gamma.ppf(U[:,5],3,scale=100*p["traffic_scale"]); d["DistHighway"]=-5.0*np.log(1-U[:,6])
    Z2=_ar_latent(n,4,phi,rng)
    d["PopDensity"]=np.exp(7+Z2[:,0]); d["Temp"]=15+8*Z2[:,1]; d["Pressure"]=1013+8*Z2[:,2]; d["IndustryIdx"]=50+10*p["industry"]+10*Z2[:,3]
    Zn=_ar_latent(n,10,phi,rng)
    for i in range(1,11): d[f"Noise{i}"]=Zn[:,i-1]
    return pd.DataFrame(d)[ALL_FEATS]

_r=gen_raw(200000,"Normal",np.random.default_rng(12345)); REF={f:(float(_r[f].mean()),float(_r[f].std())) for f in TRUE}; del _r
def true_prob(X,sc):
    p=SCENARIOS[sc]; z=lambda f:(X[f].values-REF[f][0])/REF[f][1]
    lo=p["b0"]+p["signal"]*(sum(COEF[f]*z(f) for f in TRUE)+BETA_SQ*z("GreenSpace")**2); return 1/(1+np.exp(-lo))
def simulate(n,sc,rng,min_per_class=2):
    for _ in range(500):
        X=gen_raw(n,sc,rng); pr=true_prob(X,sc); y=rng.binomial(1,pr)
        if min(y.sum(),n-y.sum())>=min_per_class: return X,y,pr
    raise RuntimeError("iki sınıf üretilemedi")
_rows=[]
for _sc in SCENARIOS:
    _X,_y,_pr=simulate(50000,_sc,np.random.default_rng(1),50)
    _rows.append(dict(scenario=_sc,prevalence=float(_y.mean()),oracle_AUC=float(roc_auc_score(_y,_pr)),rho=SCENARIOS[_sc]["rho"],signal=SCENARIOS[_sc]["signal"],phi=SCENARIOS[_sc].get("phi",0.0)))
DIAG=pd.DataFrame(_rows)


def reliability_bins(y,p,bins=10):
    p=np.asarray(p,float); y=np.asarray(y,float); idx=np.clip((p*bins).astype(int),0,bins-1)
    return (np.bincount(idx,minlength=bins).astype(float),np.bincount(idx,weights=p,minlength=bins),np.bincount(idx,weights=y,minlength=bins))
def ece_score(y,p,bins=10):
    c,sp,sy=reliability_bins(y,p,bins); return float(np.abs(sp-sy).sum()/len(y))
def pair_jaccard(B):
    B=B.astype(float); inter=B@B.T; s=B.sum(1); union=s[:,None]+s[None,:]-inter
    with np.errstate(all="ignore"): J=np.where(union>0,inter/union,np.nan)
    return J[np.triu_indices(len(B),1)]
def top_idx(imp,k=5):
    o=np.argsort(-imp,kind="stable")[:k]; return [int(i) for i in o if imp[i]>1e-12]
def jacc(a,b):
    u=a|b; return len(a&b)/len(u) if u else np.nan


class SVMPlatt:
    """RBF-SVM + sırayı bozmayan sigmoid; sigmoid çapraz doğrulamalı (out-of-fold) karar değerleriyle uydurulur."""
    def __init__(self,**kw): self.kw=kw
    def fit(self,X,y):
        self.svm=SVC(kernel="rbf",**self.kw).fit(X,y)
        dtr=self.svm.decision_function(X); d=None
        k=min(5,int(min(y.sum(),len(y)-y.sum())))
        if k>=2:
            try: d=cross_val_predict(SVC(kernel="rbf",**self.kw),X,y,cv=StratifiedKFold(k,shuffle=True,random_state=0),method="decision_function")
            except Exception: d=None
        if d is None: d=dtr
        self.lr=LogisticRegression(C=1.0).fit(np.asarray(d).reshape(-1,1),y)
        if self.lr.coef_[0,0]<=0: self.lr=LogisticRegression(C=1.0).fit(np.asarray(dtr).reshape(-1,1),y)
        return self
    def predict_proba(self,X): return self.lr.predict_proba(self.svm.decision_function(X).reshape(-1,1))

def get_models(cfg,seed):
    big=cfg["big"]
    return {
     "LogReg":(LogisticRegression(max_iter=3000),{"C":[0.01,0.1,1,10]}),
     "XGBoost":(XGBClassifier(eval_metric="logloss",n_jobs=1,random_state=seed,verbosity=0,tree_method="hist"),
                {"max_depth":[2,3,4,5] if big else [3],"learning_rate":[0.05,0.1],"n_estimators":[100,300] if big else [100]}),
     "CatBoost":(CatBoostClassifier(verbose=0,thread_count=1,random_seed=seed,allow_writing_files=False),
                {"depth":[3,5] if big else [3],"iterations":[100,300] if big else [100],"learning_rate":[0.05,0.1]}),
     "RandomForest":(RandomForestClassifier(n_estimators=300 if big else 100,n_jobs=1,random_state=seed),{"max_features":["sqrt",0.5],"min_samples_leaf":[1,5]}),
     "ExtraTrees":(ExtraTreesClassifier(n_estimators=300 if big else 100,n_jobs=1,random_state=seed),{"max_features":["sqrt",0.5],"min_samples_leaf":[1,5]}),
     "SVM":(SVC(kernel="rbf"),{"C":[0.1,0.5,1,2,5,10] if big else [0.1,1,10],"gamma":["scale",0.01,0.1] if big else ["scale"]}),
     "ANN":(MLPClassifier(max_iter=1000,random_state=seed),{"hidden_layer_sizes":[(3,),(6,),(10,)] if big else [(5,),(10,)],"alpha":[1e-3,1e-1,1.0]}),
    }
def tune_fit(est,grid,X,y,seed):
    k=min(3,int(min(y.sum(),len(y)-y.sum())))
    if k>=2:
        cv=StratifiedKFold(k,shuffle=True,random_state=seed)
        try:
            if isinstance(est,SVC):
                gs=GridSearchCV(SVC(kernel="rbf"),grid,scoring="roc_auc",cv=cv,error_score=np.nan).fit(X,y)
                return SVMPlatt(**gs.best_params_).fit(X,y),gs.best_params_
            gs=GridSearchCV(est,grid,scoring="neg_brier_score",cv=cv,error_score=np.nan).fit(X,y); return gs.best_estimator_,gs.best_params_
        except Exception: pass
    if isinstance(est,SVC): return SVMPlatt().fit(X,y),{}
    est.fit(X,y); return est,{}
def elastic_net_select(Xs,y,seed):
    p=Xs.shape[1]; k=min(5,int(min(y.sum(),len(y)-y.sum())))
    if k<2: return np.arange(p),"cv_infeasible"
    cv=StratifiedKFold(k,shuffle=True,random_state=seed); en=None; err=""
    for kw in (dict(penalty="elasticnet"),dict()):
        try: en=LogisticRegressionCV(Cs=10,cv=cv,solver="saga",l1_ratios=[0.5],scoring="neg_log_loss",max_iter=3000,tol=1e-3,random_state=seed,**kw).fit(Xs,y); break
        except Exception as e: en=None; err=f"{type(e).__name__}"
    if en is None: return np.arange(p),"error_"+err
    sel=np.where(np.abs(en.coef_[0])>1e-8)[0]
    return (sel,"ok") if len(sel) else (np.arange(p),"empty_selection")
TREE_MODELS={"XGBoost","CatBoost","RandomForest","ExtraTrees"}
TREE_INT_MODELS=TREE_MODELS-{"XGBoost"}
def _to2d(v):
    if isinstance(v,list): v=v[1] if len(v)>1 else v[0]
    v=np.asarray(v); return v[:,:,1] if v.ndim==3 else v
def shap_importance(name,model,Xtr,Xe,cfg,seed):
    if Xtr.shape[1]==1: return np.ones(1),"trivial"
    kind=None; v=None
    try:
        if name in TREE_MODELS: v=shap.TreeExplainer(model).shap_values(Xe,check_additivity=False); kind="TreeSHAP"
        elif name=="LogReg": v=shap.LinearExplainer(model,Xtr).shap_values(Xe); kind="LinearSHAP"
    except Exception: kind=None
    if kind is None:
        bg=shap.sample(Xtr,min(cfg["bg"],len(Xtr)),random_state=seed)
        ke=shap.KernelExplainer(lambda d:model.predict_proba(d)[:,1],bg)
        try: v=ke.shap_values(Xe,nsamples=cfg["kshap"],l1_reg=0,silent=True)
        except Exception: v=ke.shap_values(Xe,nsamples=cfg["kshap"],silent=True)
        kind="KernelSHAP"
    return np.abs(_to2d(v)).mean(0),kind
def shap_tree_int(model,Xtr,Xe,cfg,seed):
    """TreeSHAP, interventional biçim (arka plan verisiyle): korelasyonlu değişkenlere kredi paylaşımını sınamak için."""
    bg=shap.sample(Xtr,min(100,len(Xtr)),random_state=seed)
    v=shap.TreeExplainer(model,data=bg,feature_perturbation="interventional").shap_values(Xe,check_additivity=False)
    return np.abs(_to2d(v)).mean(0)
def lime_importance(model,Xtr,Xe,cfg,seed):
    p=Xtr.shape[1]
    if p==1: return np.ones(1)
    ex=LimeTabularExplainer(Xtr,mode="classification",discretize_continuous=False,feature_selection="none",random_state=seed,class_names=[0,1])
    imp=np.zeros(p)
    for x in Xe:
        e=ex.explain_instance(x,model.predict_proba,num_features=p,num_samples=cfg["lime_n"],labels=(1,))
        for j,w in e.as_map()[1]: imp[j]+=abs(w)
    return imp/len(Xe)

def core_run(Xtr_all,ytr,Xte_all,yte,fnames,cfg,seed,rng,selfchk=False):
    Pn=len(fnames); sel_en,en_status=elastic_net_select(Xtr_all,ytr,seed)
    eidx=rng.choice(len(yte),min(cfg["n_expl"],len(yte)),replace=False); perf=[];imps=[];rel=[];errs=[]
    for mode,sel in (("FULL",np.arange(Pn)),("EN",sel_en)):
        if mode=="EN" and len(sel)==Pn:
            perf+=[{**r,"mode":"EN"} for r in perf if r["mode"]=="FULL"]
            imps+=[{**r,"mode":"EN"} for r in imps if r["mode"]=="FULL"]
            rel +=[{**r,"mode":"EN"} for r in rel  if r["mode"]=="FULL"]; continue
        Xtr=Xtr_all[:,sel]; Xte=Xte_all[:,sel]
        for mname,(est,grid) in get_models(cfg,seed).items():
            stage="fit"
            try:
                t0=time.time(); model,bp=tune_fit(est,grid,Xtr,ytr,seed); pte=model.predict_proba(Xte)[:,1]
                row=dict(mode=mode,model=mname,n_feat=len(sel),AUC=roc_auc_score(yte,pte),Brier=brier_score_loss(yte,pte),
                         LogLoss=log_loss(yte,np.clip(pte,1e-6,1-1e-6)),ECE=ece_score(yte,pte),best_params=str(bp))
                c,sp,sy=reliability_bins(yte,pte); Xe=Xte[eidx]
                stage="shap"; sh,kind=shap_importance(mname,model,Xtr,Xe,cfg,seed)
                stage="lime"; li=lime_importance(model,Xtr,Xe,cfg,seed)
                selfj=np.nan
                if selfchk: selfj=jacc(set(top_idx(li)),set(top_idx(lime_importance(model,Xtr,Xe,cfg,seed+7919))))
                row["sec"]=time.time()-t0
            except Exception as e:
                errs.append(dict(mode=mode,model=mname,stage=stage,msg=f"{type(e).__name__}: {str(e)[:300]}")); continue
            perf.append(row); rel.append(dict(mode=mode,model=mname,count=c,sum_p=sp,sum_y=sy))
            for meth,v,ex in (("SHAP",sh,kind),("LIME",li,"LIME")):
                full=np.zeros(Pn,np.float32); full[sel]=v
                imps.append(dict(mode=mode,model=mname,method=meth,explainer=ex,imp=full,self_jacc=selfj if meth=="LIME" else np.nan))
            if cfg.get("tree_int") and mname in TREE_INT_MODELS:
                try:
                    si=shap_tree_int(model,Xtr,Xe,cfg,seed); full=np.zeros(Pn,np.float32); full[sel]=si
                    imps.append(dict(mode=mode,model=mname,method="SHAPint",explainer="TreeSHAP_int",imp=full,self_jacc=np.nan))
                except Exception as e: errs.append(dict(mode=mode,model=mname,stage="shap_int",msg=f"{type(e).__name__}: {str(e)[:300]}"))
    if not imps: raise RuntimeError("Hiçbir model/açıklayıcı çalışmadı. Hatalar: "+str(errs[:3]))
    en=np.zeros(Pn,bool); en[sel_en]=True; return perf,imps,rel,en,en_status,errs

def run_sim_task(a,cfg):
    sc,n,rep,seed,selfchk=a; rng=np.random.default_rng(seed)
    X,y,_=simulate(n,sc,rng); Xt,yt,_=simulate(cfg["n_test"],sc,rng,20); s=StandardScaler().fit(X.values)
    perf,imps,rel,en,st,errs=core_run(s.transform(X.values),y,s.transform(Xt.values),yt,ALL_FEATS,cfg,seed,rng,selfchk)
    return dict(perf=perf,imps=imps,rel=rel,en=en,st=st,errs=errs,fn=ALL_FEATS,meta=dict(scenario=sc,n=n,rep=rep))


def _fetch(name,uid,zip_url,csv_name,local_hint):
    cache=f"{DAT}/{name}.csv"
    if os.path.exists(cache): return pd.read_csv(cache)
    df=None
    if os.path.exists(local_hint): df=pd.read_csv(local_hint)
    if df is None:
        try:
            from ucimlrepo import fetch_ucirepo; df=fetch_ucirepo(id=uid).data.original.copy()
        except Exception: df=None
    if df is None:
        import urllib.request,zipfile
        z=zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(zip_url,timeout=90).read())); df=pd.read_csv(z.open(csv_name))
    df.to_csv(cache,index=False); return df
def load_forest_fires():
    df=_fetch("forestfires",162,"https://archive.ics.uci.edu/static/public/162/forest+fires.zip","forestfires.csv","/content/forestfires.csv")
    mm={m:i for i,m in enumerate(["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"],1)}
    df["month_num"]=df["month"].astype(str).str.lower().map(mm); feats=["FFMC","DMC","DC","ISI","temp","RH","wind","rain","month_num"]
    return df[feats].astype(float).values,(df["area"]>0).astype(int).values,feats,df
def load_beijing():
    df=_fetch("beijing_pm25",381,"https://archive.ics.uci.edu/static/public/381/beijing+pm2+5+data.zip","PRSA_data_2010.1.1-2014.12.31.csv","/content/PRSA_data_2010.1.1-2014.12.31.csv")
    df=df.rename(columns={"pm2.5":"pm25"}).dropna(subset=["pm25"]).reset_index(drop=True)
    wd=pd.get_dummies(df["cbwd"],prefix="wd").astype(float); Xd=pd.concat([df[["DEWP","TEMP","PRES","Iws","Is","Ir","month","hour"]],wd],axis=1)
    df["date"]=pd.to_datetime(df[["year","month","day"]]); return Xd.values.astype(float),(df["pm25"]>=75).astype(int).values,list(Xd.columns),df
Xf,yf,ff_feats,df_ff=load_forest_fires(); Xb,yb,bj_feats,df_bj=load_beijing()
print("Forest Fires:",Xf.shape,"prevalans",round(float(yf.mean()),3),"| Beijing:",Xb.shape,"prevalans",round(float(yb.mean()),3))
DATASETS={"BeijingPM25":dict(X=Xb,y=yb,feats=bj_feats,split="temporal",year=df_bj["year"].values,sizes=PROF["bj"]),
          "ForestFires":dict(X=Xf,y=yf,feats=ff_feats,split="random",sizes=PROF["ff"])}
DATASETS_ALL={**DATASETS,"BeijingBlock":dict(X=Xb,y=yb,feats=bj_feats,split="temporal",year=df_bj["year"].values,sampling="block",sizes=PROF["bj"])}
K_SHADOW=5; SHADOW=[f"Shadow{i+1}" for i in range(K_SHADOW)]
def subsample(X,y,n,rng):
    if n is None or n>=len(y): return X,y
    for _ in range(200):
        i=rng.choice(len(y),n,replace=False)
        if min(y[i].sum(),n-y[i].sum())>=3: return X[i],y[i]
    raise RuntimeError("alt örneklem üretilemedi")
def block_subsample(X,y,n,rng):
    """Zaman sıralı havuzdan ardışık n gözlemlik bir blok (seri bağımlılığı korur)."""
    N=len(y)
    if n is None or n>=N: return X,y
    for _ in range(300):
        s=int(rng.integers(0,N-n+1)); yy=y[s:s+n]
        if min(yy.sum(),n-yy.sum())>=3: return X[s:s+n],yy
    raise RuntimeError("blok örneklem üretilemedi")
def real_task(a,cfg):
    name,size,rep,seed=a; D=DATASETS_ALL[name]; rng=np.random.default_rng(seed); X,y=D["X"],D["y"]
    if D["split"]=="temporal":
        tr=D["year"]<=2013; Xp,yp=X[tr],y[tr]; Xa,ya=X[~tr],y[~tr]; ti=rng.choice(len(ya),min(1000,len(ya)),replace=False); Xt,yt=Xa[ti],ya[ti]
    else: Xp,Xt,yp,yt=train_test_split(X,y,test_size=0.3,stratify=y,random_state=int(seed%2**31))
    Xtr,ytr=(block_subsample(Xp,yp,size,rng) if D.get("sampling")=="block" else subsample(Xp,yp,size,rng))
    cols=rng.choice(Xtr.shape[1],K_SHADOW,replace=True)
    Xtr=np.hstack([Xtr,np.column_stack([rng.permutation(Xtr[:,c]) for c in cols])]); Xt=np.hstack([Xt,np.column_stack([rng.permutation(Xt[:,c]) for c in cols])])
    fn=D["feats"]+SHADOW; s=StandardScaler().fit(Xtr)
    perf,imps,rel,en,st,errs=core_run(s.transform(Xtr),ytr,s.transform(Xt),yt,fn,cfg,seed,rng,False)
    meta=dict(dataset=name,n=len(ytr),rep=rep,n_test=len(yt),prev_train=float(ytr.mean()),en_shadow_rate=float(en[-K_SHADOW:].mean()),en_real_rate=float(en[:-K_SHADOW].mean()))
    return dict(perf=perf,imps=imps,rel=rel,en=en,st=st,errs=errs,fn=fn,meta=meta)


def smoke():
    print("\n--- DUMAN TESTİ: 1 simülasyon + AR + tree_int + 3 gerçek veri görevi ---"); cfg=dict(PROF,lime_n=100,kshap=100,n_expl=4,bg=8,big=False); t0=time.time()
    r=run_sim_task(("Normal",60,0,12345,True),cfg)
    pf=pd.DataFrame(r["perf"]); print(pf[["mode","model","n_feat","AUC","ECE","sec"]].round(3).to_string(index=False))
    print("Açıklayıcı türleri:",sorted(set(i["explainer"] for i in r["imps"])),"| Elastic Net durumu:",r["st"],"| EN seçilen değişken:",int(r["en"].sum()))
    if r["errs"]: print("⚠ Model/açıklayıcı hataları:"); [print("  ",e) for e in r["errs"]]
    xa=add_metrics(pd.DataFrame(r["imps"]),ALL_FEATS,CAT_SIM,CATS_SIM,TRUE_W)
    assert {"mode","TFRR","tau_all","share_decoy"}<=set(xa.columns),"metrik sütunları eksik"
    Xar,_,_=simulate(2000,"AR0.8",np.random.default_rng(5)); ac=float(np.corrcoef(Xar["Noise1"].values[1:],Xar["Noise1"].values[:-1])[0,1])
    print(f"AR0.8 üreteci: Noise1 lag-1 otokorelasyonu = {ac:.2f} (≈0.80 beklenir)"); assert 0.6<ac<0.95,"AR üreteci beklenmedik"
    r2=run_sim_task(("AR0.8",80,0,4242,False),dict(cfg,tree_int=True))
    print("AR + tree_int yöntemleri:",sorted(set((i["method"],i["explainer"]) for i in r2["imps"])),"| hata:",len(r2["errs"]))
    if r2["errs"]: [print("  ",e) for e in r2["errs"][:4]]
    for nm,sz in (("ForestFires",50),("BeijingPM25",100),("BeijingBlock",100)):
        rr=real_task((nm,sz,0,777),cfg); print(f"Gerçek veri {nm}: {len(rr['perf'])} model sonucu, {len(rr['imps'])} açıklama, hata={len(rr['errs'])}")
        if rr["errs"]: print("  ilk hata:",rr["errs"][0])
    print(f"--- Duman testi tamam ({(time.time()-t0)/60:.1f} dk) ---\n")


def row_metrics(imp,fnames,cat,cats,true_w=None,k=5):
    t=top_idx(imp,k); out={"top_size":len(t)}
    for c in cats: out[f"share_{c}"]=np.mean([cat[fnames[i]]==c for i in t]) if t else np.nan
    if true_w is not None:
        ti=np.where(true_w>0)[0]; out["TFRR"]=len(set(t)&set(ti))/len(ti)
        out["tau_all"]=stats.kendalltau(imp,true_w)[0]; out["tau_true5"]=stats.kendalltau(imp[ti],true_w[ti])[0]
    return out
def add_metrics(df,fnames,cat,cats,true_w=None):
    if df.empty: raise RuntimeError("Açıklama tablosu boş (imps)")
    M=pd.DataFrame([row_metrics(v,fnames,cat,cats,true_w) for v in df["imp"]],index=df.index)
    df=pd.concat([df,M],axis=1); df["top"]=[frozenset(top_idx(v)) for v in df["imp"]]; return df
def stability_table(df,by):
    rows=[]
    for key,g in df.groupby(by):
        key=key if isinstance(key,tuple) else (key,); R=len(g)
        if R<2: continue
        M=np.vstack(g["imp"].values).astype(float); B=np.zeros(M.shape,np.uint8)
        for i,t in enumerate(g["top"]): B[i,list(t)]=1
        Rk=np.apply_along_axis(stats.rankdata,1,M); Rk=Rk-Rk.mean(1,keepdims=True); sd=np.sqrt((Rk**2).sum(1))
        with np.errstate(all="ignore"): C=(Rk@Rk.T)/np.outer(sd,sd)
        rows.append({**dict(zip(by,key)),"jaccard":np.nanmean(pair_jaccard(B)),"spearman":np.nanmean(C[np.triu_indices(R,1)]),"R":R})
    return pd.DataFrame(rows)
def boot_ci(x,B=500,seed=0):
    x=np.asarray(x,float); x=x[~np.isnan(x)]
    if len(x)<2: return np.nan,np.nan
    return tuple(np.percentile(np.random.default_rng(seed).choice(x,(B,len(x))).mean(1),[2.5,97.5]))
def summarize(df,by,cols):
    out=[]
    for key,g in df.groupby(by):
        key=key if isinstance(key,tuple) else (key,); r=dict(zip(by,key))
        for c in cols:
            if c in g: lo,hi=boot_ci(g[c].values); r.update({f"{c}_mean":g[c].mean(),f"{c}_sd":g[c].std(),f"{c}_ciLo":lo,f"{c}_ciHi":hi})
        r["N"]=len(g); out.append(r)
    return pd.DataFrame(out)
def rank_biserial(d):
    d=np.asarray(d); d=d[d!=0]
    if len(d)==0: return 0.0
    r=stats.rankdata(np.abs(d)); return (r[d>0].sum()-r[d<0].sum())/r.sum()
def _wil(d): return 1.0 if np.allclose(d,0) else stats.wilcoxon(d)[1]
def paired_tests(df,metric,gcols,ucols):
    rows=[]
    for key,g in df.groupby(gcols):
        key=key if isinstance(key,tuple) else (key,); u=g.groupby(ucols+["method"])[metric].mean().unstack("method").dropna()
        if len(u)<5 or not {"SHAP","LIME"}<=set(u.columns): continue
        d=u["SHAP"]-u["LIME"]; lo,hi=boot_ci(d.values)
        rows.append({**dict(zip(gcols,key)),"metric":metric,"units":len(u),"mean_SHAP":u["SHAP"].mean(),"mean_LIME":u["LIME"].mean(),"mean_diff":d.mean(),
                     "diff_ciLo":lo,"diff_ciHi":hi,"rank_biserial":rank_biserial(d),"p":_wil(d)})
    d2=pd.DataFrame(rows)
    if len(d2): d2["p_holm"]=multipletests(d2["p"],method="holm")[1]
    return d2
def model_tests(df,metric,higher=True):
    fr=[];pw=[]
    for (mode,n),g in df.groupby(["mode","n"]):
        u=g.groupby(["scenario","rep","model"])[metric].mean().unstack("model").dropna()
        if len(u)<5 or u.shape[1]<3: continue
        chi,p=stats.friedmanchisquare(*[u[c] for c in u.columns]); best=u.mean().idxmax() if higher else u.mean().idxmin()
        fr.append(dict(mode=mode,n=n,metric=metric,best=best,friedman_chi2=chi,p=p))
        for m in u.columns:
            if m!=best:
                d=u[best]-u[m]; pw.append(dict(mode=mode,n=n,metric=metric,best=best,other=m,median_diff=d.median(),p=_wil(d)))
    pw=pd.DataFrame(pw)
    if len(pw): pw["p_holm"]=multipletests(pw["p"],method="holm")[1]
    return pd.DataFrame(fr),pw
def fmt_pivot(df,idx,mean,sd,cols="n"):
    d=df.copy(); d["v"]=d[mean].map("{:.3f}".format)+" ("+d[sd].map("{:.3f}".format)+")"
    return d.pivot_table(index=idx,columns=cols,values="v",aggfunc="first").reset_index()


def worker(kind,key,args,cfg,ckdir):
    import warnings,os,pickle,traceback; warnings.filterwarnings("ignore"); path=f"{ckdir}/{kind}__{key}.pkl"
    if os.path.exists(path): return True
    try:
        fn=real_task if kind in ("real","block") else run_sim_task
        try:
            from threadpoolctl import threadpool_limits
            with threadpool_limits(1): res=fn(args,cfg)
        except ImportError: res=fn(args,cfg)
        pickle.dump(res,open(path+".tmp","wb")); os.replace(path+".tmp",path); return True
    except Exception: return traceback.format_exc()[-700:]
def _seed(*k): return int(np.random.SeedSequence([abs(int(x)) if not isinstance(x,str) else zlib.crc32(x.encode())%10**6 for x in k]).generate_state(1)[0])
def tasks_for(kind):
    if kind=="sim":    return sorted([(f"{sc}_{n}_{r}",(sc,n,r,_seed(7,sc,n,r),r<PROF["self_reps"])) for sc in MAIN_SC for n in PROF["sizes"] for r in range(PROF["reps"])],key=lambda t:t[1][1])
    if kind=="sens":   return sorted([(f"{sc}_{n}_{r}",(sc,n,r,_seed(9,sc,n,r),r<PROF["self_reps"])) for sc in SENS for n in PROF["sens_sizes"] for r in range(PROF["sens_reps"])],key=lambda t:t[1][1])
    if kind=="budget": return sorted([(f"{sc}_{n}_{r}",(sc,n,r,_seed(7,sc,n,r),False)) for sc in MAIN_SC for n in PROF["b_sizes"] for r in range(PROF["b_reps"])],key=lambda t:t[1][1])
    if kind=="ar":     return sorted([(f"{sc}_{n}_{r}",(sc,n,r,_seed(13,sc,n,r),False)) for sc in AR_SC for n in PROF["ar_sizes"] for r in range(PROF["ar_reps"])],key=lambda t:t[1][1])
    if kind=="block":  return sorted([(f"BeijingBlock_{sz}_{r}",("BeijingBlock",sz,r,_seed(17,"BeijingBlock",sz,r))) for sz in PROF["bj"] for r in range(PROF["block_reps"])],key=lambda t:t[1][1])
    return sorted([(f"{nm}_{sz}_{r}",(nm,sz,r,_seed(11,nm,sz or 0,r))) for nm,D in DATASETS.items() for sz in D["sizes"] for r in range(PROF["real_reps"])],key=lambda t:(t[1][1] or 10**9))
FAILS=[]
def run_stage(kind):
    tasks=tasks_for(kind); ck=CKD[kind]; todo=[t for t in tasks if not os.path.exists(f"{ck}/{kind}__{t[0]}.pkl")]
    print(f"\n[{kind}] toplam {len(tasks)} | tamamlanmış {len(tasks)-len(todo)} | kalan {len(todo)}  (kayıt: {ck})")
    if not todo: return True
    t0=time.time(); step=max(N_JOBS*2,4); done=0
    with Parallel(n_jobs=N_JOBS) as par:
        for i in range(0,len(todo),step):
            if TIME_BUDGET_MIN and (time.time()-T_START)/60>TIME_BUDGET_MIN:
                print("⏸ Zaman bütçesi doldu. Hücreyi tekrar çalıştır; kaldığı yerden devam eder."); return False
            ch=todo[i:i+step]; out=par(delayed(worker)(kind,k,a,CFGK[kind],ck) for k,a in ch)
            bad=[(k,o) for (k,_),o in zip(ch,out) if o is not True]; FAILS.extend(bad); done+=len(ch); el=time.time()-t0
            if bad and len(bad)==len(ch) and i==0: raise RuntimeError(f"İlk grup tamamen başarısız oldu:\n{bad[0][1]}")
            print(f"  {done}/{len(todo)} | geçen {el/60:.1f} dk | tahmini kalan {el/done*(len(todo)-done)/60:.1f} dk (küçük n önce koşar; başta iyimser) | hatalı görev: {len(FAILS)}",flush=True); gc.collect()
    return True
def load_stage(kind): return [pickle.load(open(p,"rb")) for k,_ in tasks_for(kind) if os.path.exists(p:=f"{CKD[kind]}/{kind}__{k}.pkl")]
def build(res):
    perf=[];imps=[];rel=[];en=[];errs=[]
    for r in res:
        m=r["meta"]; perf+=[{**m,**x} for x in r["perf"]]; imps+=[{**m,**x} for x in r["imps"]]; rel+=[{**m,**x} for x in r["rel"]]
        errs+=[{**m,**x} for x in r["errs"]]; en.append({**m,"en_status":r["st"],"en_empty":r["st"]!="ok",**dict(zip(r["fn"],r["en"]))})
    return pd.DataFrame(perf),pd.DataFrame(imps),pd.DataFrame(rel),pd.DataFrame(en),pd.DataFrame(errs)


def lg(df,y,name,col=None,row=None,hue="model",title="",hline=None):
    g=sns.relplot(data=df,x="n",y=y,hue=hue,col=col,row=row,kind="line",marker="o",errorbar=("ci",95),n_boot=200,
                  palette=(PAL if hue=="model" else None),height=3.0,aspect=1.35,facet_kws=dict(sharex=False))
    sizes=sorted(df["n"].unique())
    for ax in g.axes.flat:
        ax.set_xscale("log"); ax.minorticks_off(); ax.set_xlabel("Training sample size (n)")
        if len(sizes)<=8: ax.set_xticks(sizes); ax.set_xticklabels(sizes,rotation=45)
        if hline is not None: ax.axhline(hline,color="grey",ls=":")
    g.fig.suptitle(title,y=1.03); sv(g.fig,name)
def heat(df,val,name,title,modes=("FULL",),fmt=".2f"):
    for mode in modes:
        meths=sorted(df["method"].unique()) if "method" in df.columns else [None]
        fig,axs=plt.subplots(1,len(meths),figsize=(4.6*len(meths),3.6),squeeze=False)
        for ax,me in zip(axs[0],meths):
            g=df[df["mode"]==mode]
            if me: g=g[g["method"]==me]
            M=g.pivot_table(index="model",columns="n",values=val).reindex([m for m in MODEL_ORDER if m in g.model.unique()])
            sns.heatmap(M,annot=True,fmt=fmt,cmap="viridis",ax=ax,cbar=False); ax.set_title(f"{me or ''} ({mode})")
        fig.suptitle(title,y=1.05); sv(fig,f"{name}_{mode}")


def analysis():
    E=pd.DataFrame(); SUMTXT=[]; ROB={}
    perf_sum=xai_sum=xai_meth=stab=stab_sum=stab_mod=lime_self=expl_sum=en_freq=en_sum=E
    t_tfrr=t_tau=t_tau5=t_noise=t_jac=fr_auc=pw_auc=fr_ece=pw_ece=fr_tf=pw_tf=mixed=thr=diffs=sens_sum=sens_perf=errs_sim=E
    r_perf_sum=r_xai_sum=r_xai_mod=r_stab=r_stab_sum=r_en=r_test=r_agree=r_agree_sum=r_imp=errs_real=E
    rperf=rxai=E; CH={}

    perf,imps,rel,en,errs_sim=build(load_stage("sim")); print(f"Simülasyon: perf {len(perf)} satır, açıklama {len(imps)} satır, model hatası {len(errs_sim)}")
    xai=add_metrics(imps,ALL_FEATS,CAT_SIM,CATS_SIM,TRUE_W); perf=perf[perf["AUC"].notna()].copy()
    with guard("ham CSV"):
        perf.drop(columns=["best_params"],errors="ignore").to_csv(f"{CSV}/sim_perf_raw.csv",index=False)
        xai.drop(columns=["imp","top"]).to_csv(f"{CSV}/sim_xai_raw_noimp.csv",index=False)
    if len(errs_sim): show("Simülasyonda yakalanan model/açıklayıcı hataları (özet)",errs_sim.groupby(["model","stage","msg"]).size().reset_index(name="adet"))

    with guard("özet tablolar"):
        perf_sum=summarize(perf,["mode","n","model"],["AUC","Brier","ECE","LogLoss"])
        cols=["TFRR","share_true","share_decoy","share_nuisance","share_noise","tau_all","tau_true5"]
        xai_sum=summarize(xai,["mode","n","model","method"],cols); xai_meth=summarize(xai,["mode","n","method"],cols)
    with guard("kararlılık"):
        stab=stability_table(xai,["mode","scenario","n","model","method"])
        stab_sum=stab.groupby(["mode","n","method"])[["jaccard","spearman"]].mean().reset_index()
        stab_mod=stab.groupby(["mode","n","model","method"])[["jaccard","spearman"]].mean().reset_index()
    with guard("LIME/explainer/EN"):
        lime_self=xai[xai.method=="LIME"].groupby(["mode","n","model"])["self_jacc"].mean().reset_index()
        expl_sum=xai.groupby(["mode","n","explainer"])[["TFRR","share_noise","share_decoy","tau_all"]].mean().reset_index()
        en_freq=en.groupby(["scenario","n"])[ALL_FEATS].mean().T
        en_sum=pd.DataFrame([dict(scenario=sc,n=n,TPR_true=g[TRUE].values.mean(),sel_decoy=g[DECOY].values.mean(),sel_nuisance=g[NUIS].values.mean(),
                  sel_noise=g[NOISE].values.mean(),en_empty_rate=g["en_empty"].mean()) for (sc,n),g in en.groupby(["scenario","n"])])
        print("Elastic Net durum dağılımı:",en["en_status"].value_counts().to_dict())
    with guard("testler"):
        t_tfrr=paired_tests(xai,"TFRR",["mode","n"],["scenario","rep"]); t_tau=paired_tests(xai,"tau_all",["mode","n"],["scenario","rep"])
        t_tau5=paired_tests(xai,"tau_true5",["mode","n"],["scenario","rep"]); t_noise=paired_tests(xai,"share_noise",["mode","n"],["scenario","rep"])
        rows=[]
        for (mode,n),g in stab.groupby(["mode","n"]):
            u=g.pivot_table(index=["scenario","model"],columns="method",values="jaccard").dropna()
            if len(u)>=5:
                d=u["SHAP"]-u["LIME"]; lo,hi=boot_ci(d.values)
                rows.append(dict(mode=mode,n=n,metric="jaccard",units=len(u),mean_SHAP=u["SHAP"].mean(),mean_LIME=u["LIME"].mean(),mean_diff=d.mean(),diff_ciLo=lo,diff_ciHi=hi,rank_biserial=rank_biserial(d),p=_wil(d)))
        t_jac=pd.DataFrame(rows)
        if len(t_jac): t_jac["p_holm"]=multipletests(t_jac["p"],method="holm")[1]
        parts=[x.assign(metric=m)[["mode","n","metric","mean_diff","diff_ciLo","diff_ciHi","p_holm"]] for x,m in [(t_tfrr,"TFRR"),(t_jac,"Jaccard"),(t_tau,"Kendall tau (all)"),(t_noise,"Noise share")] if len(x)]
        diffs=pd.concat(parts) if parts else E
    with guard("model testleri"):
        fr_auc,pw_auc=model_tests(perf,"AUC",True); fr_ece,pw_ece=model_tests(perf,"ECE",False); fr_tf,pw_tf=model_tests(xai,"TFRR",True)
    with guard("karma model"):
        d=xai.dropna(subset=["TFRR"]).copy(); d["dataset"]=d.scenario+"_"+d.n.astype(str)+"_"+d.rep.astype(str); d["n_cat"]=d.n.astype(str)
        md=smf.mixedlm("TFRR ~ C(model)+C(method)+C(mode)+C(n_cat)",d,groups=d["dataset"]).fit(reml=True); print(md.summary()); mixed=md.summary().tables[1].reset_index()
    with guard("eşik n"):
        thr=pd.DataFrame([dict(mode=mode,model=model,method=meth,min_n_TFRR_ge_0_90=(g[g.TFRR_mean>=0.9].n.min() if (g.TFRR_mean>=0.9).any() else np.nan))
                          for (mode,model,meth),g in xai_sum.groupby(["mode","model","method"])])
    for title,df in [("DGP tanılama (prevalans, oracle AUC)",DIAG),("Elastic Net seçim özeti",en_sum),("AUC/Brier/ECE",perf_sum),("XAI: SHAP vs LIME",xai_meth),("Kararlılık",stab_sum),
                     ("Açıklayıcı türüne göre",expl_sum),("LIME kendi-içi tutarlılık",lime_self),("TFRR testi",t_tfrr),("Jaccard testi",t_jac),("Kendall tau testi",t_tau),
                     ("Friedman AUC",fr_auc),("Friedman TFRR",fr_tf),("TFRR>=0.90 için asgari n",thr)]: show(title,df)

    with guard("S01 tasarım"):
        fig,ax=plt.subplots(figsize=(13,3.2)); ax.axis("off")
        boxes=["DGP\n(3 scenarios, 26 variables:\n5 DGP, 3 correlated decoys,\n8 nuisance, 10 noise)","Training n\n+ independent\nlarge test set","Variable set\nFULL vs Elastic Net\n(training data only)","7 models\n(LogReg, XGB, CatBoost, RF,\nExtraTrees, SVM, ANN)","XAI: SHAP & LIME\n(top-5, TFRR,\ndecoy/noise share)","Stability &\nDGP agreement\n(Jaccard, Spearman, Kendall)","Real data + robustness:\nBeijing PM2.5, Forest Fires,\nblock sampling, AR(1) DGP"]
        for i,b in enumerate(boxes):
            ax.text(0.07+i*0.145,0.5,b,ha="center",va="center",fontsize=8.5,bbox=dict(boxstyle="round,pad=0.5",fc="#eef3fb",ec="#4c72b0"),transform=ax.transAxes)
            if i<len(boxes)-1: ax.annotate("",xy=(0.07+(i+1)*0.145-0.058,0.5),xytext=(0.07+i*0.145+0.058,0.5),xycoords="axes fraction",arrowprops=dict(arrowstyle="->"))
        ax.set_title("Study design",fontweight="bold"); sv(fig,"S01_study_design")
    with guard("S02 DGP"):
        Xv=gen_raw(5000,"Normal",np.random.default_rng(3)); fig,axs=plt.subplots(1,4,figsize=(16,3.8))
        sns.heatmap(np.log(Xv[POLL]).corr(),annot=True,fmt=".2f",cmap="vlag",vmin=-1,vmax=1,ax=axs[0],cbar=False); axs[0].set_title("Pollutant correlation (log scale)")
        for ax,f in zip(axs[1:3],["WindSpeed","TrafficDensity"]): sns.histplot(Xv[f],kde=True,ax=ax); ax.set_title(f)
        sns.barplot(data=DIAG,x="scenario",y="oracle_AUC",ax=axs[3],color="#4c72b0"); axs[3].set_title("Oracle (Bayes) AUC"); axs[3].tick_params(axis="x",rotation=60); sv(fig,"S02_dgp_description")
    for m in ["AUC","ECE","Brier"]:
        with guard(f"S03 {m}"): lg(perf,m,f"S03_{m}_vs_n",col="mode",title=f"{m} vs sample size (pooled over scenarios; 95% CI)")
    with guard("S03b"): lg(perf[perf["mode"]=="FULL"],"AUC","S03b_AUC_by_scenario",col="scenario",title="AUC by scenario (FULL)")
    with guard("S04 reliability"):
        rows=[]
        for (mode,n,model),g in rel.groupby(["mode","n","model"]):
            c=np.sum(np.vstack(g["count"]),0); sp_=np.sum(np.vstack(g["sum_p"]),0); sy_=np.sum(np.vstack(g["sum_y"]),0)
            rows+=[dict(mode=mode,n=n,model=model,pred=sp_[b]/c[b],obs=sy_[b]/c[b]) for b in range(len(c)) if c[b]>0]
        rl=pd.DataFrame(rows)
        for mode in ["FULL","EN"]:
            ns=sorted(rl.n.unique()); fig,axs=plt.subplots(1,len(ns),figsize=(3.6*len(ns),3.8),sharey=True,squeeze=False)
            for ax,n in zip(axs[0],ns):
                for m in MODEL_ORDER:
                    g=rl[(rl["mode"]==mode)&(rl.n==n)&(rl.model==m)]
                    if len(g): ax.plot(g.pred,g.obs,marker="o",ms=3,label=m,color=PAL[m])
                ax.plot([0,1],[0,1],"r--"); ax.set_title(f"n={n}"); ax.set_xlabel("Predicted probability")
            axs[0][0].set_ylabel("Observed frequency"); axs[0][-1].legend(fontsize=7); fig.suptitle(f"Reliability curves ({mode})",y=1.04); sv(fig,f"S04_reliability_{mode}")
    with guard("S05 Elastic Net"):
        fig,axs=plt.subplots(1,3,figsize=(15,8),sharey=True)
        for ax,sc in zip(axs,MAIN_SC):
            sns.heatmap(en_freq[sc],annot=True,fmt=".1f",cmap="viridis",vmin=0,vmax=1,ax=ax,cbar=False,annot_kws={"size":7}); ax.set_title(sc)
            for t in ax.get_yticklabels():
                if t.get_text() in TRUE: t.set_color("#E8710A"); t.set_fontweight("bold")
        fig.suptitle("Elastic Net selection frequency (orange = variable in the DGP)",y=1.0); sv(fig,"S05_elasticnet_selection")
    with guard("S06 TFRR"): lg(xai,"TFRR","S06_TFRR",col="method",row="mode",title="TFRR: share of the 5 DGP variables recovered in top-5",hline=0.9)
    with guard("S06b bileşim"):
        colors=["#E8710A","#f2c185","#9aa0a6","#3c4043"]; fig,axs=plt.subplots(2,2,figsize=(11,8),sharey=True)
        for i,mode in enumerate(["FULL","EN"]):
            for j,meth in enumerate(["SHAP","LIME"]):
                xai[(xai["mode"]==mode)&(xai.method==meth)].groupby("n")[[f"share_{c}" for c in CATS_SIM]].mean().plot(kind="bar",stacked=True,ax=axs[i,j],color=colors,legend=(i==0 and j==1))
                axs[i,j].set_title(f"{mode} – {meth}"); axs[i,j].set_ylabel("Share of top-5")
        fig.suptitle("Top-5 composition: DGP / correlated decoys / nuisance / noise",y=1.0); sv(fig,"S06b_top5_composition")
    with guard("S07 jaccard"): lg(stab,"jaccard","S07_jaccard",col="method",row="mode",title="Top-5 Jaccard stability across replications")
    with guard("S07b spearman"): lg(stab,"spearman","S07b_spearman",col="method",row="mode",title="Mean pairwise Spearman across replications")
    with guard("S08 kendall"): lg(xai,"tau_all","S08_kendall_all",col="method",row="mode",title="Kendall tau-b: importance vs DGP |beta| (all variables)")
    with guard("S08b kendall5"): lg(xai,"tau_true5","S08b_kendall_true5",col="method",row="mode",title="Kendall tau within the 5 DGP variables")
    with guard("S09 heat TFRR"): heat(xai_sum.rename(columns={"TFRR_mean":"v"}),"v","S09_heat_TFRR","Mean TFRR (model x n)",modes=("FULL","EN"))
    with guard("S09b heat Jaccard"): heat(stab_mod.rename(columns={"jaccard":"v"}),"v","S09b_heat_Jaccard","Mean Jaccard (model x n)")
    with guard("S09c heat AUC"): heat(perf_sum.rename(columns={"AUC_mean":"v"}),"v","S09c_heat_AUC","Mean AUC (model x n)",fmt=".3f")
    with guard("S10 forest"):
        d=diffs[diffs["mode"]=="FULL"]; mets=list(d.metric.unique()); fig,axs=plt.subplots(1,len(mets),figsize=(4.2*len(mets),3.8),sharey=True,squeeze=False)
        for ax,me in zip(axs[0],mets):
            g=d[d.metric==me].sort_values("n"); lo=np.clip(g.mean_diff-g.diff_ciLo,0,None); hi=np.clip(g.diff_ciHi-g.mean_diff,0,None)
            ax.errorbar(g.mean_diff,range(len(g)),xerr=[lo,hi],fmt="o",capsize=3); ax.axvline(0,color="r",ls="--")
            ax.set_yticks(range(len(g))); ax.set_yticklabels([f"n={n}" for n in g.n]); ax.set_title(me); ax.set_xlabel("SHAP - LIME (95% CI)")
        fig.suptitle("Paired difference SHAP - LIME (FULL)",y=1.04); sv(fig,"S10_forest_shap_minus_lime")
    with guard("S11 AUC vs TFRR"):
        pr=pd.concat([perf.groupby(["mode","n","model"])["AUC"].mean(),xai.groupby(["mode","n","model"])["TFRR"].mean()],axis=1).dropna().reset_index()
        fig,axs=plt.subplots(1,2,figsize=(12,4.5),sharey=True)
        for ax,mode in zip(axs,["FULL","EN"]):
            g=pr[pr["mode"]==mode]; sns.scatterplot(data=g,x="AUC",y="TFRR",hue="model",size="n",sizes=(30,250),palette=PAL,ax=ax); ax.set_title(mode)
            if len(g)>3: ax.text(0.02,0.95,f"Spearman rho={stats.spearmanr(g.AUC,g.TFRR)[0]:.2f}",transform=ax.transAxes)
        sv(fig,"S11_AUC_vs_TFRR")
    def bars(mode,n,models=("CatBoost","SVM","LogReg"),topn=12):
        models=[m for m in models if m in xai.model.unique()]; fig,axs=plt.subplots(2,len(models),figsize=(5*len(models),7),squeeze=False)
        for i,meth in enumerate(("SHAP","LIME")):
            for j,m in enumerate(models):
                g=xai[(xai["mode"]==mode)&(xai.n==n)&(xai.model==m)&(xai.method==meth)]
                if g.empty: continue
                M=np.vstack(g.imp.values).astype(float); M=M/np.maximum(M.sum(1,keepdims=True),1e-12); mean=M.mean(0); o=np.argsort(-mean)[:topn][::-1]
                axs[i,j].barh([ALL_FEATS[k] for k in o],mean[o],xerr=M.std(0)[o],color=["#E8710A" if ALL_FEATS[k] in TRUE else "#9aa0a6" for k in o]); axs[i,j].set_title(f"{m} – {meth} (n={n})")
        sv(fig,f"S12_importance_{mode}_n{n}")
    for n in sorted({min(PROF["sizes"]),max(PROF["sizes"])}):
        for mode in ["FULL","EN"]:
            with guard(f"S12 {mode} {n}"): bars(mode,n)
    with guard("S13 LIME self"):
        ls=xai[(xai.method=="LIME")&xai.self_jacc.notna()]; fig,ax=plt.subplots(figsize=(7,4)); sns.boxplot(data=ls,x="n",y="self_jacc",hue="mode",ax=ax)
        ax.set_title("LIME self-consistency (same model, different seed)"); sv(fig,"S13_lime_self_consistency")
    with guard("S14 explainer type"):
        fig,ax=plt.subplots(figsize=(8,4)); sns.barplot(data=expl_sum[expl_sum["mode"]=="FULL"],x="n",y="TFRR",hue="explainer",ax=ax)
        ax.set_title("TFRR by explainer type (separates algorithm from model effect)"); sv(fig,"S14_explainer_type")

    if RUN_SENS:
        with guard("duyarlılık"):
            perf_s,imps_s,_,_,_=build(load_stage("sens")); xai_s=add_metrics(imps_s,ALL_FEATS,CAT_SIM,CATS_SIM,TRUE_W); perf_s=perf_s[perf_s["AUC"].notna()]
            xai_b=pd.concat([xai[(xai.scenario=="Normal")&xai.n.isin(PROF["sens_sizes"])],xai_s]); perf_b=pd.concat([perf[(perf.scenario=="Normal")&perf.n.isin(PROF["sens_sizes"])],perf_s])
            stab_s=stability_table(xai_b,["mode","scenario","n","model","method"])
            sens_sum=xai_b.groupby(["mode","scenario","n","method"])[["TFRR","share_decoy","share_noise","tau_all"]].mean().reset_index().merge(
                stab_s.groupby(["mode","scenario","n","method"])[["jaccard","spearman"]].mean().reset_index(),on=["mode","scenario","n","method"])
            sens_perf=perf_b.groupby(["mode","scenario","n"])[["AUC","ECE"]].mean().reset_index(); show("Duyarlılık: XAI",sens_sum); show("Duyarlılık: performans",sens_perf)
            nn=max(PROF["sens_sizes"]); fig,axs=plt.subplots(1,4,figsize=(17,3.8))
            for ax,me in zip(axs,["TFRR","jaccard","share_decoy","tau_all"]):
                sns.barplot(data=sens_sum[(sens_sum.n==nn)&(sens_sum["mode"]=="FULL")],x="scenario",y=me,hue="method",ax=ax); ax.set_title(me); ax.tick_params(axis="x",rotation=45)
            fig.suptitle(f"Sensitivity analysis (n={nn}, FULL)",y=1.04); sv(fig,"S15_sensitivity")

    with guard("gerçek veri analizi"):
        rperf,rimps,_,ren,errs_real=build(load_stage("real")); rperf=rperf[rperf["AUC"].notna()].copy(); RX=[]
        print(f"Gerçek veri: perf {len(rperf)}, açıklama {len(rimps)}, hata {len(errs_real)}")
        for nm,D in DATASETS.items():
            fn=D["feats"]+SHADOW; cat={**{f:"real" for f in D["feats"]},**{s:"shadow" for s in SHADOW}}; RX.append(add_metrics(rimps[rimps.dataset==nm].copy(),fn,cat,["real","shadow"]))
        rxai=pd.concat(RX); rxai["shadow_hit"]=(rxai["share_shadow"]>0).astype(float)
        r_perf_sum=summarize(rperf,["dataset","mode","n","model"],["AUC","Brier","ECE"]); r_xai_sum=summarize(rxai,["dataset","mode","n","method"],["share_shadow","shadow_hit"])
        r_xai_sum["chance_share"]=r_xai_sum["dataset"].map({nm:K_SHADOW/(len(D["feats"])+K_SHADOW) for nm,D in DATASETS.items()})
        r_xai_mod=summarize(rxai,["dataset","mode","n","model","method"],["share_shadow"]); r_stab=stability_table(rxai,["dataset","mode","n","model","method"])
        r_stab_sum=r_stab.groupby(["dataset","mode","n","method"])[["jaccard","spearman"]].mean().reset_index()
        r_en=ren.groupby(["dataset","n"])[["en_real_rate","en_shadow_rate","en_empty"]].mean().reset_index()
        r_test=paired_tests(rxai,"share_shadow",["dataset","mode","n"],["rep"])
        ag=[]
        for key,g in rxai.groupby(["dataset","mode","n","rep","model"]):
            if set(g.method)=={"SHAP","LIME"}:
                a_=g[g.method=="SHAP"].iloc[0]; b_=g[g.method=="LIME"].iloc[0]
                ag.append(dict(dataset=key[0],mode=key[1],n=key[2],rep=key[3],model=key[4],spearman_SHAP_LIME=stats.spearmanr(a_.imp,b_.imp)[0],jaccard_SHAP_LIME=jacc(a_.top,b_.top)))
        r_agree=pd.DataFrame(ag); r_agree_sum=r_agree.groupby(["dataset","mode","n"])[["spearman_SHAP_LIME","jaccard_SHAP_LIME"]].mean().reset_index()
        imp_rows=[]
        for nm,D in DATASETS.items():
            fn=D["feats"]+SHADOW; nmax=rxai[rxai.dataset==nm].n.max()
            for meth in ["SHAP","LIME"]:
                g=rxai[(rxai.dataset==nm)&(rxai["mode"]=="FULL")&(rxai.n==nmax)&(rxai.method==meth)]
                if g.empty: continue
                M=np.vstack(g.imp.values).astype(float); M=M/np.maximum(M.sum(1,keepdims=True),1e-12)
                imp_rows+=[dict(dataset=nm,n=nmax,method=meth,feature=f,mean_norm_importance=M.mean(0)[j],sd=M.std(0)[j],is_shadow=f in SHADOW) for j,f in enumerate(fn)]
        r_imp=pd.DataFrame(imp_rows).sort_values(["dataset","method","mean_norm_importance"],ascending=[True,True,False])
        for title,df in [("GERÇEK VERİ – performans",r_perf_sum),("GERÇEK VERİ – gölge değişken payı (chance_share = şans düzeyi)",r_xai_sum),("GERÇEK VERİ – SHAP vs LIME testi",r_test),("GERÇEK VERİ – kararlılık",r_stab_sum),
                         ("GERÇEK VERİ – SHAP–LIME uyumu",r_agree_sum),("GERÇEK VERİ – Elastic Net gölge seçimi",r_en),("GERÇEK VERİ – ortalama önem",r_imp)]: show(title,df,60)
    def rlines(df,y,name,title,hue="model",chance=None):
        g=sns.relplot(data=df,x="n",y=y,hue=hue,col="dataset",row="mode",kind="line",marker="o",errorbar=("ci",95),n_boot=200,palette=(PAL if hue=="model" else None),height=3.1,aspect=1.45,facet_kws=dict(sharex=False))
        for ax in g.axes.flat: ax.set_xscale("log"); ax.minorticks_off(); ax.set_xlabel("Training sample size (n)")
        if chance:
            try:
                for (r_,c_),ax in g.axes_dict.items():
                    ds=c_ if c_ in chance else (r_ if r_ in chance else None)
                    if ds is not None: ax.axhline(chance[ds],color="grey",ls=":")
            except Exception: pass
        g.fig.suptitle(title,y=1.03); sv(g.fig,name)
    with guard("R01 betimleyici"):
        fig,axs=plt.subplots(1,3,figsize=(17,4.2)); dd=df_bj.set_index("date")["pm25"].resample("W").mean()
        axs[0].plot(dd.index,dd.values,lw=0.8); axs[0].axhline(75,color="r",ls="--"); axs[0].axvspan(pd.Timestamp("2014-01-01"),dd.index.max(),color="orange",alpha=0.2)
        axs[0].set_title("Beijing PM2.5 (weekly mean); shaded = test year 2014"); axs[0].set_ylabel("ug/m3")
        sns.boxplot(data=df_bj,x="month",y="pm25",ax=axs[1],showfliers=False); axs[1].set_title("Beijing PM2.5 seasonality")
        if {"X","Y","area"}<=set(df_ff.columns):
            sc_=axs[2].scatter(df_ff["X"]+np.random.default_rng(0).uniform(-.25,.25,len(df_ff)),df_ff["Y"]+np.random.default_rng(1).uniform(-.25,.25,len(df_ff)),
                               s=15+30*np.log1p(df_ff["area"]),c=np.log1p(df_ff["area"]),cmap="YlOrRd",alpha=.7,edgecolor="k",lw=.3)
            plt.colorbar(sc_,ax=axs[2],label="log(1+area ha)"); axs[2].set_title("Forest Fires: spatial grid of Montesinho Park"); axs[2].set_xlabel("X"); axs[2].set_ylabel("Y")
        sv(fig,"R01_real_data_description")
    CH={nm:K_SHADOW/(len(D["feats"])+K_SHADOW) for nm,D in DATASETS.items()}
    with guard("R02"): rlines(rperf,"AUC","R02_AUC","Real data: AUC vs n (temporal hold-out for Beijing)")
    with guard("R03"): rlines(rperf,"ECE","R03_ECE","Real data: ECE vs n")
    with guard("R04"): rlines(rxai,"share_shadow","R04_shadow_share","Real data: share of shadow variables in top-5 (dotted = chance level)",hue="method",chance=CH)
    with guard("R05"): rlines(r_stab,"jaccard","R05_jaccard","Real data: top-5 Jaccard stability",hue="method")
    with guard("R05b"): rlines(r_stab,"spearman","R05b_spearman","Real data: pairwise Spearman stability",hue="method")
    with guard("R06"): rlines(r_agree,"spearman_SHAP_LIME","R06_shap_lime_agreement","Real data: SHAP–LIME rank agreement",hue="model")
    for nm in DATASETS:
        with guard(f"R07 {nm}"):
            fig,axs=plt.subplots(1,2,figsize=(12,5.5))
            for ax,meth in zip(axs,["SHAP","LIME"]):
                g=r_imp[(r_imp.dataset==nm)&(r_imp.method==meth)].sort_values("mean_norm_importance")
                ax.barh(g.feature,g.mean_norm_importance,xerr=g.sd,color=["#9aa0a6" if s else "#4c72b0" for s in g.is_shadow]); ax.set_title(f"{nm} – {meth} (grey = shadow)")
            sv(fig,f"R07_importance_{nm}")
    with guard("R08"):
        pv=r_imp.pivot_table(index=["dataset","feature"],columns="method",values="mean_norm_importance").reset_index(); fig,axs=plt.subplots(1,2,figsize=(10,4.3))
        for ax,nm in zip(axs,DATASETS):
            g=pv[pv.dataset==nm]; ax.scatter(g.SHAP,g.LIME,c=["#9aa0a6" if f.startswith("Shadow") else "#4c72b0" for f in g.feature])
            for _,r_ in g.iterrows(): ax.annotate(r_.feature,(r_.SHAP,r_.LIME),fontsize=7)
            mx=g[["SHAP","LIME"]].values.max(); ax.plot([0,mx],[0,mx],"r--"); ax.set_title(nm); ax.set_xlabel("SHAP importance"); ax.set_ylabel("LIME importance")
        sv(fig,"R08_shap_vs_lime_importance")
    with guard("MAIN kompozit"):
        fig,axs=plt.subplots(2,2,figsize=(11,8))
        sns.lineplot(data=xai[xai["mode"]=="FULL"],x="n",y="TFRR",hue="method",marker="o",errorbar=("ci",95),n_boot=200,ax=axs[0,0]); axs[0,0].set_title("(a) Simulation: TFRR")
        sns.lineplot(data=stab[stab["mode"]=="FULL"],x="n",y="jaccard",hue="method",marker="o",errorbar=("ci",95),n_boot=200,ax=axs[0,1]); axs[0,1].set_title("(b) Simulation: Jaccard stability")
        for ax,nm,lab in zip(axs[1],DATASETS,["(c)","(d)"]):
            sns.lineplot(data=rxai[(rxai.dataset==nm)&(rxai["mode"]=="FULL")],x="n",y="share_shadow",hue="method",marker="o",errorbar=("ci",95),n_boot=200,ax=ax)
            ax.axhline(CH[nm],color="grey",ls=":"); ax.set_title(f"{lab} {nm}: shadow share in top-5")
        for ax in axs.flat: ax.set_xscale("log"); ax.set_xlabel("Training sample size (n)")
        sv(fig,"MAIN_composite_figure")


    if RUN_BUDGET:
        with guard("ROB budget"):
            perf_bu,imps_bu,_,_,errs_bu=build(load_stage("budget")); perf_bu=perf_bu[perf_bu["AUC"].notna()].copy()
            xai_bu=add_metrics(imps_bu,ALL_FEATS,CAT_SIM,CATS_SIM,TRUE_W)
            print(f"Bütçe aşaması: {len(perf_bu)} model, {len(xai_bu)} açıklama, hata {len(errs_bu)}")
            if len(errs_bu): show("Bütçe aşaması hataları (özet)",errs_bu.groupby(["model","stage","msg"]).size().reset_index(name="adet"))
            chk=perf_bu.merge(perf[["scenario","n","rep","mode","model","AUC"]],on=["scenario","n","rep","mode","model"],suffixes=("_high","_std"))
            ROB["budget_pairing_check"]=pd.DataFrame([dict(matched_rows=len(chk),max_abs_AUC_diff=float((chk.AUC_high-chk.AUC_std).abs().max()) if len(chk) else np.nan)])
            lo_=xai[xai.scenario.isin(MAIN_SC)&xai.n.isin(PROF["b_sizes"])&(xai.rep<PROF["b_reps"])]
            xb=pd.concat([lo_.assign(budget="standard"),xai_bu.assign(budget="high")],ignore_index=True)
            cols_=["TFRR","share_decoy","share_noise","tau_all"]
            ROB["budget_summary"]=summarize(xb,["budget","mode","n","method"],cols_)
            stb=stability_table(xb,["budget","mode","scenario","n","model","method"])
            ROB["budget_stability"]=stb.groupby(["budget","mode","n","method"])[["jaccard","spearman"]].mean().reset_index()
            rows=[]
            for (meth,mode,n),g in xb[xb.method.isin(["SHAP","LIME"])].groupby(["method","mode","n"]):
                u=g.pivot_table(index=["scenario","rep","model"],columns="budget",values="TFRR").dropna()
                if len(u)<5 or not {"standard","high"}<=set(u.columns): continue
                dd=u["high"]-u["standard"]; l_,h_=boot_ci(dd.values)
                rows.append(dict(method=meth,mode=mode,n=n,units=len(u),mean_standard=u["standard"].mean(),mean_high=u["high"].mean(),mean_diff=dd.mean(),diff_ciLo=l_,diff_ciHi=h_,p=_wil(dd)))
            ROB["budget_effect_TFRR"]=pd.DataFrame(rows)
            hi2=xai_bu[xai_bu.method.isin(["SHAP","LIME"])]
            ROB["budget_SHAPvsLIME_TFRR"]=paired_tests(hi2,"TFRR",["mode","n"],["scenario","rep"])
            ROB["budget_SHAPvsLIME_decoy"]=paired_tests(hi2,"share_decoy",["mode","n"],["scenario","rep"])
            stb_hi=stability_table(hi2,["mode","scenario","n","model","method"]); rows=[]
            for (mode,n),g in stb_hi.groupby(["mode","n"]):
                u=g.pivot_table(index=["scenario","model"],columns="method",values="jaccard").dropna()
                if len(u)>=5:
                    dd=u["SHAP"]-u["LIME"]; l_,h_=boot_ci(dd.values)
                    rows.append(dict(mode=mode,n=n,units=len(u),mean_SHAP=u["SHAP"].mean(),mean_LIME=u["LIME"].mean(),mean_diff=dd.mean(),diff_ciLo=l_,diff_ciHi=h_,p=_wil(dd)))
            ROB["budget_SHAPvsLIME_Jaccard"]=pd.DataFrame(rows)
            trees=["CatBoost","RandomForest","ExtraTrees"]
            ROB["budget_tree_variants"]=summarize(xai_bu[xai_bu.model.isin(trees)],["mode","n","method"],["TFRR","share_decoy","share_noise"])
            for k_ in ["budget_pairing_check","budget_summary","budget_stability","budget_effect_TFRR","budget_SHAPvsLIME_TFRR","budget_SHAPvsLIME_Jaccard","budget_tree_variants"]: show("SAĞLAMLIK – "+k_,ROB[k_])
            g_=xb[(xb["mode"]=="FULL")&xb.method.isin(["SHAP","LIME"])].copy(); g_["variant"]=g_.method+" – "+g_.budget
            fig,axs=plt.subplots(1,3,figsize=(15,3.9))
            for ax,me,tt in zip(axs,["TFRR","share_decoy","share_noise"],["TFRR","Decoy share in top-5","Noise share in top-5"]):
                sns.barplot(data=g_,x="n",y=me,hue="variant",errorbar=("ci",95),n_boot=200,ax=ax); ax.set_title(tt); ax.set_xlabel("Training sample size (n)")
                if me!="TFRR" and ax.get_legend(): ax.get_legend().remove()
            fig.suptitle("Explanation budget: standard (LIME 400 / KernelSHAP 600 / 12 obs.) vs high (1000 / 1500 / 24 obs.), FULL",y=1.04); sv(fig,"ROB1_budget_effect")
            tv=xai_bu[xai_bu.model.isin(trees)&(xai_bu["mode"]=="FULL")].copy(); tv["variant"]=tv.method.map({"SHAP":"TreeSHAP (path-dependent)","SHAPint":"TreeSHAP (interventional)","LIME":"LIME"})
            fig,axs=plt.subplots(1,3,figsize=(15,3.9))
            for ax,me,tt in zip(axs,["TFRR","share_decoy","share_noise"],["TFRR","Decoy share in top-5","Noise share in top-5"]):
                sns.barplot(data=tv,x="n",y=me,hue="variant",errorbar=("ci",95),n_boot=200,ax=ax); ax.set_title(tt); ax.set_xlabel("Training sample size (n)")
                if me!="TFRR" and ax.get_legend(): ax.get_legend().remove()
            fig.suptitle("CatBoost, random forest, extremely randomized trees (high budget, FULL): TreeSHAP variants vs LIME",y=1.04); sv(fig,"ROB2_tree_variants")

     if RUN_BLOCK:
        with guard("ROB block"):
            rperf_b,rimps_b,_,_,errs_blk=build(load_stage("block")); rperf_b=rperf_b[rperf_b["AUC"].notna()].copy()
            fnb=DATASETS_ALL["BeijingBlock"]["feats"]+SHADOW; catb={**{f:"real" for f in DATASETS_ALL["BeijingBlock"]["feats"]},**{s:"shadow" for s in SHADOW}}
            rxb=add_metrics(rimps_b.copy(),fnb,catb,["real","shadow"]); rxb["shadow_hit"]=(rxb["share_shadow"]>0).astype(float)
            xx=pd.concat([rxai[rxai.dataset=="BeijingPM25"].assign(sampling="random"),rxb.assign(sampling="block")],ignore_index=True)
            pp=pd.concat([rperf[rperf.dataset=="BeijingPM25"].assign(sampling="random"),rperf_b.assign(sampling="block")],ignore_index=True)
            ROB["block_shadow"]=summarize(xx,["sampling","mode","n","method"],["share_shadow","shadow_hit"])
            ROB["block_perf"]=summarize(pp,["sampling","mode","n","model"],["AUC","ECE"])
            stb_blk=stability_table(xx,["sampling","mode","n","model","method"])
            ROB["block_stability"]=stb_blk.groupby(["sampling","mode","n","method"])[["jaccard","spearman"]].mean().reset_index()
            ROB["block_class_prevalence"]=pp.groupby(["sampling","n"])["prev_train"].mean().reset_index()
            for k_ in ["block_shadow","block_stability","block_class_prevalence"]: show("SAĞLAMLIK – "+k_,ROB[k_])
            ff_=xx[xx["mode"]=="FULL"]; fig,axs=plt.subplots(1,3,figsize=(16,3.9))
            sns.lineplot(data=ff_,x="n",y="share_shadow",hue="sampling",style="method",marker="o",errorbar=("ci",95),n_boot=200,ax=axs[0]); axs[0].axhline(CH.get("BeijingPM25",0.29),color="grey",ls=":"); axs[0].set_title("Shadow share in top-5")
            sns.lineplot(data=pp[pp["mode"]=="FULL"],x="n",y="AUC",hue="sampling",marker="o",errorbar=("ci",95),n_boot=200,ax=axs[1]); axs[1].set_title("Test AUC (2014)")
            sns.lineplot(data=stb_blk[stb_blk["mode"]=="FULL"],x="n",y="jaccard",hue="sampling",style="method",marker="o",errorbar=("ci",95),n_boot=200,ax=axs[2]); axs[2].set_title("Top-5 Jaccard across replications")
            for ax in axs: ax.set_xscale("log"); ax.set_xlabel("Training sample size (n)")
            fig.suptitle("Beijing PM2.5: random hours vs contiguous time blocks (FULL)",y=1.04); sv(fig,"ROB3_block_vs_random")

     if RUN_AR:
        with guard("ROB AR"):
            perf_a,imps_a,_,_,errs_a=build(load_stage("ar")); perf_a=perf_a[perf_a["AUC"].notna()].copy(); xai_a=add_metrics(imps_a,ALL_FEATS,CAT_SIM,CATS_SIM,TRUE_W)
            for dfx in (xai_a,perf_a): dfx["phi"]=dfx.scenario.str.replace("AR","",regex=False).astype(float)
            iidx=xai[(xai.scenario=="Normal")&xai.n.isin(PROF["ar_sizes"])].assign(phi=0.0); iidp=perf[(perf.scenario=="Normal")&perf.n.isin(PROF["ar_sizes"])].assign(phi=0.0)
            xa=pd.concat([iidx,xai_a],ignore_index=True); pa=pd.concat([iidp,perf_a],ignore_index=True)
            for dfx in (xa,pa): dfx["phi_label"]="phi="+dfx.phi.astype(str)
            xa["n_eff"]=xa.n*(1-xa.phi)/(1+xa.phi)
            cols_=["TFRR","share_decoy","share_noise","tau_all"]
            ROB["ar_xai"]=summarize(xa,["phi","mode","n","method"],cols_)
            ROB["ar_perf"]=pa.groupby(["phi","mode","n"])[["AUC","ECE"]].mean().reset_index()
            stab_a=stability_table(xa,["phi","mode","scenario","n","model","method"]); stab_a["phi_label"]="phi="+stab_a.phi.astype(str)
            ROB["ar_stability"]=stab_a.groupby(["phi","mode","n","method"])[["jaccard","spearman"]].mean().reset_index()
            ROB["ar_SHAPvsLIME_TFRR"]=paired_tests(xai_a,"TFRR",["phi","mode","n"],["scenario","rep"])
            ROB["ar_DGP_diagnostics"]=DIAG[DIAG.scenario.isin(["Normal"]+AR_SC)]
            for k_ in ["ar_xai","ar_perf","ar_stability","ar_SHAPvsLIME_TFRR","ar_DGP_diagnostics"]: show("SAĞLAMLIK – "+k_,ROB[k_])
            fa=xa[xa["mode"]=="FULL"]
            lg(fa,"TFRR","ROB4_ar_TFRR",col="method",hue="phi_label",title="TFRR with temporally autocorrelated covariates (FULL)",hline=0.9)
            fig,ax=plt.subplots(figsize=(6.5,4.2)); sns.lineplot(data=fa,x="n_eff",y="TFRR",hue="phi_label",style="method",marker="o",errorbar=("ci",95),n_boot=200,ax=ax)
            ax.set_xscale("log"); ax.set_xlabel("Effective sample size n(1−φ)/(1+φ)"); ax.set_title("TFRR against effective sample size (FULL)"); sv(fig,"ROB5_ar_neff")
            lg(stab_a[stab_a["mode"]=="FULL"],"jaccard","ROB6_ar_jaccard",col="method",hue="phi_label",title="Top-5 Jaccard stability with autocorrelated covariates (FULL)")
            lg(fa,"share_decoy","ROB7_ar_decoy",col="method",hue="phi_label",title="Decoy share in top-5 with autocorrelated covariates (FULL)")

     cfg_df=pd.DataFrame([dict(CODE_VER=CODE_VER,PROFILE=PROFILE,SVM_probability="Platt sigmoid on out-of-fold decision values (SVMPlatt)",TreeSHAP="default path-dependent; interventional variant for CatBoost/RF/ExtraTrees in budget analysis",**{k:str(v) for k,v in PROF.items()})])
    with guard("Excel simülasyon"):
        xai_export=pd.concat([xai.drop(columns=["imp","top"]),pd.DataFrame(np.vstack(xai.imp),columns=ALL_FEATS,index=xai.index).add_prefix("imp_")],axis=1)
        tabs={"config":cfg_df,"dgp_diagnostics":DIAG,"perf_raw":perf.drop(columns=["best_params"],errors="ignore"),"perf_summary":perf_sum,"xai_raw":xai_export,
          "xai_summary_model":xai_sum,"xai_summary_method":xai_meth,"explainer_type":expl_sum,"stability_raw":stab,"stability_summary":stab_sum,"stability_by_model":stab_mod,
          "lime_self":lime_self,"elasticnet_freq":en_freq.reset_index() if len(en_freq) else E,"elasticnet_summary":en_sum,"threshold_n":thr,"test_TFRR":t_tfrr,"test_tauAll":t_tau,
          "test_tauTrue5":t_tau5,"test_noiseShare":t_noise,"test_Jaccard":t_jac,"friedman_AUC":fr_auc,"pairwise_AUC":pw_auc,"friedman_ECE":fr_ece,"pairwise_ECE":pw_ece,
          "friedman_TFRR":fr_tf,"pairwise_TFRR":pw_tf,"mixed_TFRR":mixed,"SENS_xai":sens_sum,"SENS_perf":sens_perf,"model_errors":errs_sim}
        if len(perf_sum): tabs["TABLE_AUC"]=fmt_pivot(perf_sum[perf_sum["mode"]=="FULL"],["model"],"AUC_mean","AUC_sd"); tabs["TABLE_ECE"]=fmt_pivot(perf_sum[perf_sum["mode"]=="FULL"],["model"],"ECE_mean","ECE_sd")
        if len(xai_sum): tabs["TABLE_TFRR"]=fmt_pivot(xai_sum[xai_sum["mode"]=="FULL"],["model","method"],"TFRR_mean","TFRR_sd")
        save_tables(tabs,f"{RES}/simulation_results_{PROFILE}.xlsx","sim")
    with guard("Excel gerçek veri"):
        tabs={"config":cfg_df,"perf_raw":rperf.drop(columns=["best_params"],errors="ignore"),"perf_summary":r_perf_sum,"shadow_summary":r_xai_sum,"shadow_by_model":r_xai_mod,"shadow_test":r_test,
              "xai_raw":rxai.drop(columns=["imp","top"]) if len(rxai) else E,"stability_raw":r_stab,"stability_summary":r_stab_sum,"agreement_raw":r_agree,"agreement":r_agree_sum,"importance_mean":r_imp,
              "elasticnet_shadow":r_en,"model_errors":errs_real}
        if len(r_perf_sum): tabs["TABLE_AUC"]=fmt_pivot(r_perf_sum[r_perf_sum["mode"]=="FULL"],["dataset","model"],"AUC_mean","AUC_sd")
        save_tables(tabs,f"{RES}/real_data_results_{PROFILE}.xlsx","real")
    with guard("Excel sağlamlık"):
        if ROB: save_tables({"config":cfg_df,**ROB},f"{RES}/robustness_results_{PROFILE}.xlsx","rob")

     pd.set_option("display.width",250,"display.max_columns",60,"display.float_format",lambda v:f"{v:.3f}")
    def P_(title,obj):
        s=f"\n##### {title}\n"+(obj.to_string() if hasattr(obj,"to_string") else str(obj)); print(s); SUMTXT.append(s)
    with guard("özet metni"):
        P_("DGP",DIAG.to_string(index=False)); P_("AUC model x n",perf_sum.pivot_table(index=["mode","model"],columns="n",values="AUC_mean"))
        P_("ECE model x n",perf_sum.pivot_table(index=["mode","model"],columns="n",values="ECE_mean")); P_("TFRR yontem x n",xai_meth.pivot_table(index=["mode","method"],columns="n",values="TFRR_mean"))
        P_("TFRR model x yontem x n (FULL)",xai_sum[xai_sum["mode"]=="FULL"].pivot_table(index=["model","method"],columns="n",values="TFRR_mean"))
        P_("Top-5 bilesimi",xai_meth[["mode","n","method","share_true_mean","share_decoy_mean","share_nuisance_mean","share_noise_mean"]].to_string(index=False))
        P_("Kararlilik",stab_sum.to_string(index=False)); P_("Kendall tau",xai_meth[["mode","n","method","tau_all_mean","tau_true5_mean"]].to_string(index=False))
        P_("Aciklayici turu",expl_sum.to_string(index=False)); P_("Test TFRR",t_tfrr.to_string(index=False)); P_("Test Jaccard",t_jac.to_string(index=False)); P_("Test tau",t_tau.to_string(index=False))
        P_("Esik n",thr.to_string(index=False)); P_("Elastic Net",en_sum.to_string(index=False)); P_("DUYARLILIK",sens_sum.to_string(index=False))
        P_("GERCEK VERI AUC",r_perf_sum.pivot_table(index=["dataset","mode","model"],columns="n",values="AUC_mean")); P_("GERCEK VERI golge payi (chance_share = sans duzeyi)",r_xai_sum[["dataset","mode","n","method","share_shadow_mean","share_shadow_sd","shadow_hit_mean","chance_share","N"]].to_string(index=False))
        P_("GERCEK VERI test",r_test.to_string(index=False)); P_("GERCEK VERI kararlilik",r_stab_sum.to_string(index=False)); P_("GERCEK VERI SHAP-LIME uyumu",r_agree_sum.to_string(index=False))
        P_("GERCEK VERI Elastic Net",r_en.to_string(index=False)); P_("GERCEK VERI ilk 8 degisken",r_imp.groupby(["dataset","method"]).head(8).to_string(index=False))
    with guard("özet metni (sağlamlık)"):
        for k_,v_ in ROB.items(): P_("SAGLAMLIK "+k_,v_.to_string(index=False))
    open(f"{RES}/SUMMARY_paste_to_chat.txt","w").write("\n".join(SUMTXT))
    log=f"Kod {CODE_VER} profil {PROFILE}\nGörev hataları: {len(FAILS)}\nAtlanan bloklar: {[l for l,_ in GUARD_ERR]}\n\n"+"\n\n".join(f"[{l}]\n{t}" for l,t in GUARD_ERR)
    open(f"{RES}/run_log.txt","w").write(log)


if RUN_SMOKE: smoke()
ok=run_stage("sim"); ok=ok and run_stage("real")
if RUN_SENS and ok:   ok=run_stage("sens")
if RUN_BLOCK and ok:  ok=run_stage("block")
if RUN_AR and ok:     ok=run_stage("ar")
if RUN_BUDGET and ok: ok=run_stage("budget")
if FAILS: print(f"\n⚠ Başarısız görev sayısı: {len(FAILS)}. İlk hata:\n{FAILS[0][1]}")
if not ok:
    print("\nİşlem tamamlanmadı (zaman bütçesi). Hücreyi tekrar çalıştır; rapor tüm görevler bitince üretilir.")
else:
    print(f"\n✅ Hesaplama tamam ({(time.time()-T_START)/60:.1f} dk). Rapor üretiliyor...")
    analysis()
    n_fig=len([f for f in os.listdir(FIG) if f.endswith(".png")]); n_csv=len(os.listdir(CSV))
    print(f"\n✅ BİTTİ. Şekil: {n_fig} PNG | CSV: {n_csv} | klasör: {RES}")
    if GUARD_ERR: print("⚠ Atlanan bloklar:",[l for l,_ in GUARD_ERR],"-> ayrıntı: run_log.txt")
    shutil.make_archive(f"/content/RESULTS_{PROFILE}","zip",RES)
    if DELETE_CHECKPOINTS_AFTER: shutil.rmtree(CK_NEW,ignore_errors=True)
    try:
        from google.colab import files; files.download(f"/content/RESULTS_{PROFILE}.zip")
    except Exception: print("ZIP:",f"/content/RESULTS_{PROFILE}.zip")